In [1]:
# Notebook'u sci5-wbc ana klasöründe açıp Restart & Run All seç.
PROJECT_ROOT_OVERRIDE = ""  # otomatik bulmazsa r"C:\Users\sarah\Desktop\sci5-wbc"
GENERATE_FIGURES = True

import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (f1_score, confusion_matrix,
                             precision_recall_fscore_support)
from IPython.display import display

start=Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
candidates=[start,*start.parents]
R=next((p for p in candidates if (p/'review_results_external_consistency_v13'/'external_consistency_decision_v13.json').is_file()),None)
if R is None:
    raise FileNotFoundError('V13 proje kökü bulunamadı. PROJECT_ROOT_OVERRIDE ayarla ve sci5-wbc içinden çalıştır.')
R=R.resolve()
OUT=R/'review_results_external_per_class_v14';OUT.mkdir(parents=True,exist_ok=True)
V13=R/'review_results_external_consistency_v13'
V10=R/'review_results_ret_found_only_v10'
V6=R/'review_results_external_BNE_SNE_v6'
DATASETS={'PBC':(10298,[1218,3117,1214,1420,3329]),'Raabin_Train_TestA':(14514,[301,1066,3461,795,8891])}
MODELS=('convnext_tiny','resnet50','retfound_fullft')
PRETTY={'convnext_tiny':'ConvNeXt-Tiny','resnet50':'ResNet-50','retfound_fullft':'RETFound'}
SEEDS=(42,1,7)
CLASSES=('BA','EO','LY','MO','N')
LABELS=list(range(5))
FULL=('BA','BL','BNE','EO','LY','MMY','MO','MY','PC','PLY','PMY','SNE','VLY')
KEEP=[FULL.index(x) for x in ('BA','EO','LY','MO','SNE')]
BNE=FULL.index('BNE'); SNE=FULL.index('SNE')

metadata=json.loads((V13/'external_consistency_decision_v13.json').read_text(encoding='utf-8'))
verified=pd.read_csv(V13/'external_per_seed_verified_v13.csv')
assert metadata.get('all_18_cached_runs_verified') is True and metadata.get('verified_runs')==18 and metadata.get('issues_count')==0, 'V13 18/18 PASS değil'
assert len(verified)==18 and (verified.status=='PASS_CACHED_EXTERNAL').all(), 'V13 seed durumları tam doğrulanmış değil'
print('Proje:',R)
print('V13 denetimi: 18/18 PASS; GPU kullanılmayacak')

Proje: C:\Users\sarah\Desktop\scı5-wbc
V13 denetimi: 18/18 PASS; GPU kullanılmayacak


## 1. Düzeltilmiş tahminleri V13 ile tekrar eşleştir

Kayıtlar yalnızca doğrulanmış V6/V10 çıktılarından okunur. `paths.txt` ve `true.npy` karşılaştırması görüntü sırasının tutarlılığını korur. Baseline ve BNE+SNE kararları 13 sınıflı olasılıklardan bağımsız yeniden üretilir.

In [2]:
per_records=[]
confusion_records=[]
issues=[]
reference={}
run_count=0

for dataset,(n_expected,class_expected) in DATASETS.items():
    for model in MODELS:
        for seed in SEEDS:
            try:
                base=((V10/dataset/f'seed{seed}') if model=='retfound_fullft'
                      else (V6/dataset/model/f'seed{seed}'))
                def load(name):
                    f=base/name
                    if not f.is_file():raise FileNotFoundError(str(f))
                    return np.load(f,allow_pickle=False)
                y=load('true_5class.npy')
                pred0=load('pred_baseline_5class.npy')
                pred1=load('pred_BNEplusSNE_5class.npy')
                q13=load('probabilities_13class.npy')
                path_file=base/'paths.txt'
                if not path_file.is_file():raise FileNotFoundError(str(path_file))
                paths=path_file.read_text(encoding='utf-8').splitlines()
                if not (y.shape==pred0.shape==pred1.shape==(n_expected,) and q13.shape==(n_expected,13) and len(paths)==n_expected):
                    raise ValueError('Boyut veya dosya sırası uzunluğu uyuşmadı')
                if not np.isin(y,LABELS).all() or not np.isin(pred0,LABELS).all() or not np.isin(pred1,LABELS).all():
                    raise ValueError('Beş sınıf dışı etiket/tahmin var')
                if not np.array_equal(np.bincount(y,minlength=5),np.asarray(class_expected)):
                    raise ValueError('Gerçek sınıf destekleri eski değerlendirmeyle farklı')
                if not np.isfinite(q13).all() or np.max(np.abs(q13.sum(1)-1))>2e-5:
                    raise ValueError('13-sınıflı olasılıklar geçersiz')
                # Baseline: yalnız BA/EO/LY/MO/SNE; alternatif: BNE + SNE.
                q0=q13[:,KEEP]
                q1=q0.copy();q1[:,4]+=q13[:,BNE]
                if not np.array_equal(q0.argmax(1),pred0):
                    raise ValueError('Baseline NPY, 13 sınıflı softmax ile uyuşmuyor')
                if not np.array_equal(q1.argmax(1),pred1):
                    raise ValueError('BNE+SNE NPY, 13 sınıflı softmax ile uyuşmuyor')
                ref_key=dataset
                if ref_key not in reference:
                    reference[ref_key]=(y.copy(),paths)
                else:
                    prior_y,prior_paths=reference[ref_key]
                    if not np.array_equal(y,prior_y) or paths!=prior_paths:
                        raise ValueError('Modeller arasında gerçek etiket/görüntü sırası farklı')
                v=verified[(verified.dataset==dataset)&(verified.model==model)&(verified.seed==seed)]
                if len(v)!=1:raise ValueError('V13 seed satırı bulunamadı veya tekrarlandı')
                v=v.iloc[0]
                f0=f1_score(y,pred0,labels=LABELS,average='macro',zero_division=0)
                f1=f1_score(y,pred1,labels=LABELS,average='macro',zero_division=0)
                if abs(f0-float(v.baseline_macro_f1))>1e-6 or abs(f1-float(v.bne_sne_macro_f1))>1e-6:
                    raise ValueError('Macro-F1 V13 doğrulanmış değeriyle uyuşmuyor')
                for condition,pred in [('baseline',pred0),('BNE+SNE',pred1)]:
                    prec,rec,f1class,support=precision_recall_fscore_support(y,pred,labels=LABELS,zero_division=0)
                    matrix=confusion_matrix(y,pred,labels=LABELS)
                    for k,cls in enumerate(CLASSES):
                        per_records.append({'dataset':dataset,'model':PRETTY[model],'seed':seed,
                                            'mapping':condition,'class':cls,'support':int(support[k]),
                                            'precision':float(prec[k]),'recall':float(rec[k]),
                                            'f1':float(f1class[k]),'tp':int(matrix[k,k]),
                                            'fn':int(support[k]-matrix[k,k])})
                    for i,true_class in enumerate(CLASSES):
                        for j,pred_class in enumerate(CLASSES):
                            confusion_records.append({'dataset':dataset,'model':PRETTY[model],
                                                      'seed':seed,'mapping':condition,
                                                      'true_class':true_class,'predicted_class':pred_class,
                                                      'count':int(matrix[i,j])})
                run_count+=1
                print('PASS',dataset,PRETTY[model],seed)
            except Exception as exc:
                issues.append({'dataset':dataset,'model':model,'seed':seed,
                               'error':type(exc).__name__+': '+str(exc)})
                print('REVIEW',dataset,PRETTY[model],seed,str(exc))

pd.DataFrame(issues,columns=['dataset','model','seed','error']).to_csv(OUT/'external_class_issues_v14.csv',index=False,encoding='utf-8-sig')
if issues or run_count!=18:
    (OUT/'external_class_decision_v14.json').write_text(json.dumps({'status':'REVIEW','passed':run_count,'expected':18,'issues':issues},indent=2,ensure_ascii=False),encoding='utf-8')
    raise RuntimeError(f'Rapor durduruldu: {run_count}/18 PASS, {len(issues)} hata. external_class_issues_v14.csv dosyasını gönder.')
print('\n18/18 doğru; sınıf bazlı metrikler yazılabilir.')

PASS PBC ConvNeXt-Tiny 42
PASS PBC ConvNeXt-Tiny 1
PASS PBC ConvNeXt-Tiny 7
PASS PBC ResNet-50 42
PASS PBC ResNet-50 1
PASS PBC ResNet-50 7
PASS PBC RETFound 42
PASS PBC RETFound 1
PASS PBC RETFound 7
PASS Raabin_Train_TestA ConvNeXt-Tiny 42
PASS Raabin_Train_TestA ConvNeXt-Tiny 1
PASS Raabin_Train_TestA ConvNeXt-Tiny 7
PASS Raabin_Train_TestA ResNet-50 42
PASS Raabin_Train_TestA ResNet-50 1
PASS Raabin_Train_TestA ResNet-50 7
PASS Raabin_Train_TestA RETFound 42
PASS Raabin_Train_TestA RETFound 1
PASS Raabin_Train_TestA RETFound 7

18/18 doğru; sınıf bazlı metrikler yazılabilir.


## 2. Hakeme uygun sınıf bazlı tablolar

Her modelin üç seed ortalaması ve **örnek standart sapması** ayrı verilir. TestB ile karıştırılmaz; grafikler yalnız baseline (beş-sınıf) kararına dayanır. EO/MO/LY özellikle raporlanır.

In [3]:
per=pd.DataFrame(per_records)
confusions=pd.DataFrame(confusion_records)
summary=(per.groupby(['dataset','model','mapping','class'],as_index=False)
         .agg(n_seeds=('seed','nunique'),support=('support','first'),
              mean_precision=('precision','mean'),mean_recall=('recall','mean'),
              sd_recall=('recall','std'),mean_f1=('f1','mean'),sd_f1=('f1','std')))
reviewer=summary[(summary.mapping=='baseline')&summary['class'].isin(['EO','MO','LY'])].copy()
reviewer=reviewer.sort_values(['dataset','model','class'])
confusion_summary=(confusions.groupby(['dataset','model','mapping','true_class','predicted_class'],as_index=False)
                   .agg(mean_count=('count','mean'),min_count=('count','min'),max_count=('count','max')))

per.to_csv(OUT/'external_class_per_seed_v14.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'external_class_all_5class_summary_v14.csv',index=False,encoding='utf-8-sig')
reviewer.to_csv(OUT/'reviewer_1_11_EO_MO_LY_recall_F1_v14.csv',index=False,encoding='utf-8-sig')
confusions.to_csv(OUT/'external_confusion_5x5_per_seed_v14.csv',index=False,encoding='utf-8-sig')
confusion_summary.to_csv(OUT/'external_confusion_5x5_mean_v14.csv',index=False,encoding='utf-8-sig')

print('Reviewer 1.11 — baseline:')
display(reviewer[['dataset','model','class','support','mean_recall','sd_recall','mean_f1','sd_f1']].round(4))
print('RETFound baseline — bütün 5 sınıf:')
display(summary[(summary.model=='RETFound')&(summary.mapping=='baseline')].round(4))

Reviewer 1.11 — baseline:


,dataset,model,class,support,mean_recall,sd_recall,mean_f1,sd_f1
6,PBC,ConvNeXt-Tiny,EO,3117,0.9837,0.0158,0.9905,0.0087
7,PBC,ConvNeXt-Tiny,LY,1214,0.9920,0.0067,0.9647,0.0124
8,PBC,ConvNeXt-Tiny,MO,1420,0.8958,0.0367,0.9437,0.0210
16,PBC,RETFound,EO,3117,0.8983,0.0561,0.9451,0.0303
17,PBC,RETFound,LY,1214,0.9621,0.0358,0.9572,0.0137
18,PBC,RETFound,MO,1420,0.5728,0.0240,0.7251,0.0223
26,PBC,ResNet-50,EO,3117,0.9808,0.0032,0.9897,0.0013
27,PBC,ResNet-50,LY,1214,0.9951,0.0051,0.9369,0.0189
28,PBC,ResNet-50,MO,1420,0.6810,0.1179,0.8061,0.0852
36,Raabin_Train_TestA,ConvNeXt-Tiny,EO,1066,0.4146,0.1160,0.5695,0.1192


RETFound baseline — bütün 5 sınıf:


,dataset,model,mapping,class,n_seeds,support,mean_precision,mean_recall,sd_recall,mean_f1,sd_f1
15,PBC,RETFound,baseline,BA,3,1218,0.8078,0.8870,0.0319,0.8452,0.0236
16,PBC,RETFound,baseline,EO,3,3117,0.9984,0.8983,0.0561,0.9451,0.0303
17,PBC,RETFound,baseline,LY,3,1214,0.9549,0.9621,0.0358,0.9572,0.0137
18,PBC,RETFound,baseline,MO,3,1420,0.9880,0.5728,0.0240,0.7251,0.0223
19,PBC,RETFound,baseline,N,3,3329,0.8107,0.9981,0.0010,0.8944,0.0239
45,Raabin_Train_TestA,RETFound,baseline,BA,3,301,0.5695,0.2647,0.1102,0.3285,0.0809
46,Raabin_Train_TestA,RETFound,baseline,EO,3,1066,0.7099,0.0428,0.0173,0.0774,0.0244
47,Raabin_Train_TestA,RETFound,baseline,LY,3,3461,0.6287,0.9964,0.0044,0.7691,0.0543
48,Raabin_Train_TestA,RETFound,baseline,MO,3,795,0.1859,0.0482,0.0302,0.0727,0.0447
49,Raabin_Train_TestA,RETFound,baseline,N,3,8891,0.9810,0.9341,0.0346,0.9564,0.0094


## 3. Görseller: RETFound'un düzeltilmiş karmaşıklık matrisleri ve EO/MO/LY recall

Satır-normalize matris, gerçek sınıftaki hücrelerin hangi sınıfa tahmin edildiğini gösterir. Hücre yüzdeleri üç seed matrisinin **toplamından değil, seed-bazlı satır oranlarının ortalamasından** üretilir.

In [4]:
if GENERATE_FIGURES:
    import matplotlib.pyplot as plt
    for dataset in DATASETS:
        matrices=[]
        for seed in SEEDS:
            part=confusions[(confusions.dataset==dataset)&(confusions.model=='RETFound')&
                            (confusions.seed==seed)&(confusions.mapping=='baseline')]
            table=part.pivot(index='true_class',columns='predicted_class',values='count').reindex(index=CLASSES,columns=CLASSES)
            mat=table.to_numpy(dtype=float)
            if not np.array_equal(mat.sum(axis=1),np.array(DATASETS[dataset][1])):
                raise ValueError('Confusion matrix satır destekleri uyuşmadı')
            matrices.append(mat/mat.sum(axis=1,keepdims=True))
        mat=np.mean(matrices,axis=0)
        fig,ax=plt.subplots(figsize=(7.0,6.0))
        im=ax.imshow(mat,vmin=0,vmax=1,cmap='Blues')
        ax.set_xticks(range(5),CLASSES);ax.set_yticks(range(5),CLASSES)
        ax.set_xlabel('Predicted class');ax.set_ylabel('True class')
        ax.set_title(f'Corrected RETFound — {dataset} (3-seed mean)')
        for i in range(5):
            for j in range(5):
                ax.text(j,i,f'{100*mat[i,j]:.1f}%',ha='center',va='center',
                        color='white' if mat[i,j]>=0.55 else 'black',fontsize=10)
        fig.colorbar(im,ax=ax,label='Share of actual class')
        fig.tight_layout()
        fig.savefig(OUT/f'RETFound_{dataset}_Corrected_Confusion_v14.png',dpi=220)
        plt.close(fig)
    # Model comparison for the three classes requested by Reviewer 1.11.
    for dataset in DATASETS:
        fig,ax=plt.subplots(figsize=(8.8,4.7))
        for model in PRETTY.values():
            x=reviewer[(reviewer.dataset==dataset)&(reviewer.model==model)]
            x=x.set_index('class').reindex(['EO','MO','LY'])
            ax.plot(['EO','MO','LY'],x.mean_recall.values,marker='o',label=model)
        ax.set_ylim(0,1.05);ax.set_ylabel('Class recall');ax.set_xlabel('True class')
        ax.set_title(f'External recall: EO, MO and LY — {dataset}')
        ax.grid(alpha=.2);ax.legend(loc='lower left')
        fig.tight_layout()
        fig.savefig(OUT/f'Reviewer111_{dataset}_EO_MO_LY_Recall_v14.png',dpi=220)
        plt.close(fig)
    print('Dört PNG görseli kaydedildi.')

Dört PNG görseli kaydedildi.


## 4. Son rapor ve göndermen gereken dosyalar

Çalışma sonunda `PASS_18_RUNS` görmeden sayıları makaleye aktarma. Hakem 1.11'in yanıtında **kaydedilmiş, düzeltilmiş dış test tahminlerinden** hesaplandığı belirtilmelidir. Bu testte yeni model eğitimi veya checkpoint değiştirme yok.

In [5]:
report={'status':'PASS_18_RUNS','verified_runs':run_count,'expected_runs':18,
        'datasets':{d:DATASETS[d][0] for d in DATASETS},
        'models':list(PRETTY.values()),'seeds':list(SEEDS),
        'class_order':list(CLASSES),
        'source':'Previously audited V13 outputs: V6 CNN, V10 corrected RETFound',
        'no_retraining':True,'no_new_inference':True,
        'different_testb_not_included':True,
        'files':sorted(p.name for p in OUT.glob('*'))}
(OUT/'external_class_decision_v14.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('SON DURUM:',report['status'])
print('KAYIT:',OUT)
for f in sorted(OUT.glob('*')):
    print(' -',f.name)

SON DURUM: PASS_18_RUNS
KAYIT: C:\Users\sarah\Desktop\scı5-wbc\review_results_external_per_class_v14
 - external_class_all_5class_summary_v14.csv
 - external_class_decision_v14.json
 - external_class_issues_v14.csv
 - external_class_per_seed_v14.csv
 - external_confusion_5x5_mean_v14.csv
 - external_confusion_5x5_per_seed_v14.csv
 - RETFound_PBC_Corrected_Confusion_v14.png
 - RETFound_Raabin_Train_TestA_Corrected_Confusion_v14.png
 - Reviewer111_PBC_EO_MO_LY_Recall_v14.png
 - Reviewer111_Raabin_Train_TestA_EO_MO_LY_Recall_v14.png
 - reviewer_1_11_EO_MO_LY_recall_F1_v14.csv
